# 04 - Process one leased video

Processes frames sequentially for one `work_id`/`attempt_id`. The notebook verifies lease ownership, stages and hashes the source, heartbeats through the SDK progress callback, writes immutable attempt-scoped output, and atomically publishes `committed_attempt_id`. Failures remain visible and are re-raised for the pipeline retry policy.

**After importing into Fabric:** On the configuration code cell, select **... -> Toggle parameter cell** and confirm the parameter indicator. Then attach and pin `people_counter_<environment>` as this notebook's default Lakehouse.

In [ ]:
WORK_ID = ""
ATTEMPT_ID = ""
PIPELINE_RUN_ID = ""
ACTIVITY_RUN_ID = ""
FABRIC_JOB_INSTANCE_ID = ""
WORKER_EXECUTION_ID = ""
BUNDLE_MANIFEST_SHA256 = ""
SOURCE_STORAGE_ACCOUNT = ""
SOURCE_CONTAINER = ""
SOURCE_SHORTCUT_LOCAL_ROOT = ""
DATABASE = ""
TABLE_PREFIX = "people_counter"
LEASE_MINUTES = 30
HEARTBEAT_SECONDS = 600

In [ ]:
from datetime import datetime, timedelta, timezone
from importlib.metadata import version
from pathlib import Path
from typing import Any
from urllib.parse import unquote_to_bytes, urlsplit
import hashlib
import json
import os
import random
import re
import shutil
import time

# Fabric can fail inside hf-xet's Reqwest client; use Hugging Face's HTTPS downloader instead.
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")

from delta.tables import DeltaTable
import notebookutils
from pyspark.sql import DataFrame, SparkSession, functions as F

from people_counter import (
    RFDetrBotsortConfig,
    RTDetrOsnetConfig,
    line_count_records,
    run,
    telemetry_records,
)


class LeaseLostError(RuntimeError):
    pass


class SourceValidationError(RuntimeError):
    pass


IDENTIFIER = re.compile(r"^[A-Za-z_][A-Za-z0-9_]*$")
ENCODED_SEPARATOR = re.compile(r"%(?:2f|5c)", re.IGNORECASE)
work_id = WORK_ID.strip()
attempt_id = ATTEMPT_ID.strip()
worker_execution_id = WORKER_EXECUTION_ID.strip()
if not work_id or not attempt_id or not worker_execution_id:
    raise ValueError("WORK_ID, ATTEMPT_ID, and WORKER_EXECUTION_ID are required")
lease_minutes = int(LEASE_MINUTES)
heartbeat_seconds = int(HEARTBEAT_SECONDS)
if lease_minutes < 5 or heartbeat_seconds < 30 or heartbeat_seconds >= lease_minutes * 60:
    raise ValueError("Heartbeat must be at least 30 seconds and shorter than the lease")
database = DATABASE.strip()
prefix = TABLE_PREFIX.strip()
if database and IDENTIFIER.fullmatch(database) is None:
    raise ValueError("DATABASE is not a valid identifier")
if IDENTIFIER.fullmatch(prefix) is None:
    raise ValueError("TABLE_PREFIX is not a valid identifier")


def require_text(value: object, name: str) -> str:
    if not isinstance(value, str) or not value.strip():
        raise ValueError(f"{name} must be a non-empty string")
    return value.strip()


def source_relative_path(value: object, name: str) -> str:
    text = require_text(value, name)
    parts = urlsplit(text)
    account = require_text(SOURCE_STORAGE_ACCOUNT, "SOURCE_STORAGE_ACCOUNT").lower()
    container = require_text(SOURCE_CONTAINER, "SOURCE_CONTAINER")
    expected_https_hosts = {
        f"{account}.dfs.core.windows.net",
        f"{account}.blob.core.windows.net",
    }
    if parts.query or parts.fragment:
        raise SourceValidationError(f"{name} contains unsupported URL components")
    if ENCODED_SEPARATOR.search(parts.path):
        raise SourceValidationError(f"{name} contains an encoded path separator")
    if parts.scheme.lower() == "https":
        if parts.username or parts.password or parts.port:
            raise SourceValidationError(f"{name} contains unsupported URL authority components")
        if (parts.hostname or "").lower() not in expected_https_hosts:
            raise SourceValidationError(f"{name} references an unexpected storage account")
        prefix_segments = (container,)
    elif parts.scheme.lower() in {"abfs", "abfss"}:
        expected_netloc = f"{container}@{account}.dfs.core.windows.net"
        if parts.netloc.lower() != expected_netloc:
            raise SourceValidationError(f"{name} references an unexpected ADLS filesystem")
        prefix_segments = ()
    else:
        raise SourceValidationError(f"{name} must be an Azure Storage HTTPS or ABFS URI")
    try:
        decoded = unquote_to_bytes(parts.path).decode("utf-8", "strict")
    except UnicodeDecodeError as error:
        raise SourceValidationError(f"{name} path is not valid UTF-8") from error
    segments = decoded.split("/")[1:]
    if tuple(segments[: len(prefix_segments)]) != prefix_segments:
        raise SourceValidationError(f"{name} references an unexpected container")
    relative = segments[len(prefix_segments) :]
    if not relative or any(
        segment in {"", ".", ".."}
        or "\\" in segment
        or "\x00" in segment
        or "%" in segment
        for segment in relative
    ):
        raise SourceValidationError(f"{name} contains an unsafe or empty path")
    return "/".join(relative)


def source_uri_to_shortcut_local_path(value: object, name: str) -> Path:
    root = Path(require_text(SOURCE_SHORTCUT_LOCAL_ROOT, "SOURCE_SHORTCUT_LOCAL_ROOT"))
    if not root.is_absolute() or root.parts[:4] != ("/", "lakehouse", "default", "Files"):
        raise ValueError("SOURCE_SHORTCUT_LOCAL_ROOT must be under /lakehouse/default/Files")
    return root.joinpath(*source_relative_path(value, name).split("/"))


def table(suffix: str) -> str:
    value = f"{prefix}_{suffix}"
    return f"{database}.{value}" if database else value


spark_candidate = globals().get("spark")
if not isinstance(spark_candidate, SparkSession):
    raise RuntimeError("A Fabric Spark session is required")
spark_session = spark_candidate
spark_session.conf.set("spark.sql.session.timeZone", "UTC")
work_table = table("video_work")
attempts_table = table("video_attempts")
telemetry_table = table("telemetry_attempts")
line_counts_table = table("line_count_attempts")


def retry_delta(operation, attempts: int = 6) -> None:
    for number in range(attempts):
        try:
            operation()
            return
        except Exception as error:
            message = f"{type(error).__name__}: {error}".lower()
            retryable = any(token in message for token in ("concurrent", "conflict", "changedexception"))
            if not retryable or number == attempts - 1:
                raise
            time.sleep((2 ** number) * 0.25 + random.random() * 0.5)


def utc_now() -> datetime:
    return datetime.now(timezone.utc)


def load_work() -> dict[str, Any]:
    rows = spark_session.table(work_table).where(F.col("work_id") == work_id).limit(2).collect()
    if len(rows) != 1:
        raise RuntimeError(f"Expected one work row for {work_id}, found {len(rows)}")
    return rows[0].asDict(recursive=True)


def verify_lease(work: dict[str, Any]) -> None:
    if work["status"] == "SUCCEEDED" and work["committed_attempt_id"] == attempt_id:
        return
    if work["lease_owner_attempt_id"] != attempt_id:
        raise LeaseLostError(f"Attempt {attempt_id} does not own work {work_id}")
    if work["status"] not in {"LEASED", "STAGING", "RUNNING", "WRITING"}:
        raise LeaseLostError(f"Work is not active; status={work['status']}")
    expires = work["lease_expires_at"]
    if expires is None or expires.replace(tzinfo=timezone.utc) <= utc_now():
        raise LeaseLostError("The work lease has expired")


def merge_attempt(updates: dict[str, Any]) -> None:
    current = spark_session.table(attempts_table).where(F.col("attempt_id") == attempt_id).limit(2).collect()
    if len(current) != 1:
        raise RuntimeError(f"Expected one attempt row for {attempt_id}, found {len(current)}")
    row = current[0].asDict(recursive=True)
    row.update(updates)
    source = spark_session.createDataFrame([row], spark_session.table(attempts_table).schema)
    retry_delta(
        lambda: (
            DeltaTable.forName(spark_session, attempts_table)
            .alias("t")
            .merge(
                source.alias("s"),
                (
                    "t.attempt_id = s.attempt_id AND t.capture_date = s.capture_date AND "
                    "t.worker_execution_id = s.worker_execution_id"
                ),
            )
            .whenMatchedUpdateAll()
            .execute()
        )
    )


def claim_worker_execution(work: dict[str, Any]) -> None:
    source = spark_session.createDataFrame(
        [(attempt_id, work["capture_date"], worker_execution_id)],
        "attempt_id string, capture_date date, worker_execution_id string",
    )
    retry_delta(
        lambda: (
            DeltaTable.forName(spark_session, attempts_table)
            .alias("t")
            .merge(
                source.alias("s"),
                "t.attempt_id = s.attempt_id AND t.capture_date = s.capture_date",
            )
            .whenMatchedUpdate(
                condition=(
                    "t.status = 'LEASED' AND "
                    "(t.worker_execution_id IS NULL OR t.worker_execution_id = s.worker_execution_id)"
                ),
                set={"worker_execution_id": "s.worker_execution_id"},
            )
            .execute()
        )
    )
    rows = (
        spark_session.table(attempts_table)
        .where((F.col("attempt_id") == attempt_id) & (F.col("capture_date") == work["capture_date"]))
        .limit(2)
        .collect()
    )
    if len(rows) != 1 or rows[0].worker_execution_id != worker_execution_id:
        raise LeaseLostError("Another worker execution owns this attempt")


def heartbeat(status: str, result=None, *, force: bool = False) -> None:
    now_monotonic = time.monotonic()
    if not force and now_monotonic - heartbeat.last_sent < heartbeat_seconds:
        return
    expires = utc_now() + timedelta(minutes=lease_minutes)
    source = spark_session.createDataFrame(
        [(work_id, attempt_id, work_capture_date, status, expires)],
        "work_id string, attempt_id string, capture_date date, status string, expires timestamp",
    )
    retry_delta(
        lambda: (
            DeltaTable.forName(spark_session, work_table)
            .alias("t")
            .merge(
                source.alias("s"),
                (
                    "t.work_id = s.work_id AND t.capture_date = s.capture_date AND "
                    "t.lease_owner_attempt_id = s.attempt_id"
                ),
            )
            .whenMatchedUpdate(
                condition=(
                    "t.lease_expires_at > current_timestamp() AND ("
                    "t.status = s.status OR "
                    "(t.status = 'LEASED' AND s.status = 'STAGING') OR "
                    "(t.status = 'STAGING' AND s.status = 'RUNNING') OR "
                    "(t.status = 'RUNNING' AND s.status = 'WRITING'))"
                ),
                set={
                    "status": "s.status",
                    "last_heartbeat_at": "current_timestamp()",
                    "lease_expires_at": "s.expires",
                },
            )
            .execute()
        )
    )
    current = load_work()
    verify_lease(current)
    attempt_updates = {
        "status": status,
        "last_heartbeat_at": utc_now(),
        "pipeline_run_id": PIPELINE_RUN_ID or None,
        "activity_run_id": ACTIVITY_RUN_ID or None,
        "fabric_job_instance_id": FABRIC_JOB_INSTANCE_ID or None,
        "sdk_version": version("people-counter"),
        "bundle_manifest_sha256": BUNDLE_MANIFEST_SHA256 or None,
    }
    if result is not None:
        attempt_updates["processed_frames"] = result.processed_frames
        attempt_updates["processing_seconds"] = result.processing_seconds
    merge_attempt(attempt_updates)
    heartbeat.last_sent = now_monotonic


heartbeat.last_sent = float("-inf")


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as source:
        while chunk := source.read(8 * 1024 * 1024):
            digest.update(chunk)
    return digest.hexdigest()


def stage_source(work: dict[str, Any]) -> tuple[Path, Path]:
    heartbeat("STAGING", force=True)
    merge_attempt({"staging_started_at": utc_now()})
    temporary = Path("/tmp") / f"people-counter-{attempt_id}"
    temporary.mkdir(mode=0o700, parents=False, exist_ok=False)
    shortcut_source = source_uri_to_shortcut_local_path(work["source_uri"], "source_uri")
    suffix = Path(urlsplit(work["source_uri"]).path).suffix or ".video"
    staged = temporary / f"input{suffix}"
    try:
        required_bytes = int(work["expected_size_bytes"] * 1.10)
        available_bytes = shutil.disk_usage(temporary).free
        if available_bytes < required_bytes:
            raise RuntimeError(
                f"Insufficient local staging space: required={required_bytes}, available={available_bytes}"
            )
        if not shortcut_source.is_file():
            raise FileNotFoundError(f"Shortcut source video not found: {shortcut_source}")
        shutil.copyfile(shortcut_source, staged)
        if not staged.is_file():
            raise RuntimeError(f"Could not stage shortcut video: {shortcut_source}")
        actual_size = staged.stat().st_size
        if actual_size != work["expected_size_bytes"]:
            raise SourceValidationError(
                f"Source size mismatch: expected {work['expected_size_bytes']}, got {actual_size}"
            )
        actual_sha256 = sha256_file(staged)
        if work["expected_sha256"] and actual_sha256 != work["expected_sha256"]:
            raise SourceValidationError("Source SHA-256 does not match the manifest")
        merge_attempt(
            {
                "input_sha256": actual_sha256,
                "source_size_bytes": actual_size,
            }
        )
        return staged, temporary
    except Exception:
        staged.unlink(missing_ok=True)
        temporary.rmdir()
        raise


def build_config(work: dict[str, Any], staged: Path):
    settings = json.loads(work["config_json"])

    def report_progress(result) -> None:
        heartbeat("RUNNING", result)

    common = {
        "video": staged,
        "device_variant": settings["device_variant"],
        "device": settings["device"],
        "batch_size": int(settings["batch_size"]),
        "sample_fps": settings["sample_fps"],
        "detection_threshold": float(settings["detection_threshold"]),
        "use_fp16": bool(settings["use_fp16"]),
        "line": tuple(settings["line"]) if settings["line"] else None,
        "progress_callback": report_progress,
    }
    if settings["pipeline"] == "rtdetr-osnet":
        return RTDetrOsnetConfig(**common, detector_model=settings["detector_model"])
    if settings["pipeline"] == "rfdetr-botsort":
        return RFDetrBotsortConfig(
            **common,
            camera_motion_compensation=settings["camera_motion_compensation"],
        )
    raise ValueError(f"Unsupported pipeline: {settings['pipeline']}")


def result_frames(work: dict[str, Any], result) -> tuple[DataFrame, DataFrame]:
    recorded_at = utc_now()
    common = {
        "work_id": work_id,
        "attempt_id": attempt_id,
        "camera_id": work["camera_id"],
        "location_id": work["location_id"],
        "captured_at_utc": work["captured_at_utc"],
        "capture_date": work["capture_date"],
        "recorded_at": recorded_at,
    }
    telemetry = [
        {
            **common,
            **record,
            "person_entry_at_utc": work["captured_at_utc"] + timedelta(seconds=record["entry_seconds"]),
            "person_exit_at_utc": work["captured_at_utc"] + timedelta(seconds=record["exit_seconds"]),
        }
        for record in telemetry_records(result)
    ]
    raw_line_counts = line_count_records(result)
    selected_line_counts = [
        record
        for index, record in enumerate(raw_line_counts)
        if record["frame_in_count"] > 0
        or record["frame_out_count"] > 0
        or index == len(raw_line_counts) - 1
    ]
    line_counts = [
        {
            **common,
            **record,
            "observed_at_utc": work["captured_at_utc"] + timedelta(seconds=float(record["video_seconds"])),
        }
        for record in selected_line_counts
    ]
    return (
        spark_session.createDataFrame(telemetry, spark_session.table(telemetry_table).schema),
        spark_session.createDataFrame(line_counts, spark_session.table(line_counts_table).schema),
    )


def replace_attempt_rows(table_name: str, frame: DataFrame) -> None:
    target = DeltaTable.forName(spark_session, table_name)
    retry_delta(
        lambda: target.delete(
            (F.col("capture_date") == F.lit(work_capture_date))
            & (F.col("work_id") == work_id)
            & (F.col("attempt_id") == attempt_id)
        )
    )
    if frame.take(1):
        frame.write.format("delta").mode("append").saveAsTable(table_name)


def classify_error(error: Exception) -> tuple[bool, str]:
    if isinstance(error, LeaseLostError):
        return True, "LEASE"
    if isinstance(error, (SourceValidationError, ValueError, PermissionError)):
        return False, "INPUT"
    if isinstance(error, FileNotFoundError):
        return True, "STORAGE"
    return True, "RUNTIME"


In [ ]:
work = load_work()
verify_lease(work)
work_capture_date = work["capture_date"]
if work["status"] == "SUCCEEDED" and work["committed_attempt_id"] == attempt_id:
    outcome = {"work_id": work_id, "attempt_id": attempt_id, "status": "ALREADY_SUCCEEDED"}
else:
    claim_worker_execution(work)
    staged = None
    temporary = None
    config = None
    committed = False
    try:
        staged, temporary = stage_source(work)
        config = build_config(work, staged)
        merge_attempt({"inference_started_at": utc_now(), "status": "RUNNING"})
        heartbeat("RUNNING", force=True)
        result = run(config)
        heartbeat("WRITING", result, force=True)
        merge_attempt({"writing_started_at": utc_now()})
        telemetry_frame, line_count_frame = result_frames(work, result)
        replace_attempt_rows(telemetry_table, telemetry_frame)
        replace_attempt_rows(line_counts_table, line_count_frame)
        telemetry_count = telemetry_frame.count()
        line_count = line_count_frame.count()
        final_updates = {
            "status": "SUCCEEDED",
            "completed_at": utc_now(),
            "source_duration_seconds": (
                result.total_source_frames / result.fps if result.fps > 0 else None
            ),
            "source_fps": result.fps,
            "total_source_frames": result.total_source_frames,
            "processed_frames": result.processed_frames,
            "effective_sample_fps": result.effective_sample_fps,
            "processing_seconds": result.processing_seconds,
            "distinct_people": telemetry_count,
            "line_in_count": result.line_in_count,
            "line_out_count": result.line_out_count,
            "retryable": None,
            "error_category": None,
            "error_type": None,
            "error_message": None,
        }
        merge_attempt(final_updates)
        commit_source = spark_session.createDataFrame(
            [(work_id, attempt_id, work_capture_date)],
            "work_id string, attempt_id string, capture_date date",
        )
        retry_delta(
            lambda: (
                DeltaTable.forName(spark_session, work_table)
                .alias("t")
                .merge(
                    commit_source.alias("s"),
                    (
                        "t.work_id = s.work_id AND t.capture_date = s.capture_date AND "
                        "t.lease_owner_attempt_id = s.attempt_id"
                    ),
                )
                .whenMatchedUpdate(
                    condition=(
                        "t.status = 'WRITING' AND t.committed_attempt_id IS NULL AND "
                        "t.lease_expires_at > current_timestamp()"
                    ),
                    set={
                        "status": "'SUCCEEDED'",
                        "committed_attempt_id": "s.attempt_id",
                        "completed_at": "current_timestamp()",
                        "last_heartbeat_at": "current_timestamp()",
                        "lease_owner_attempt_id": "NULL",
                        "lease_dispatcher_id": "NULL",
                        "lease_acquired_at": "NULL",
                        "lease_expires_at": "NULL",
                        "last_error_category": "NULL",
                        "last_error_type": "NULL",
                        "last_error_message": "NULL",
                    },
                )
                .execute()
            )
        )
        published = load_work()
        if published["status"] != "SUCCEEDED" or published["committed_attempt_id"] != attempt_id:
            raise LeaseLostError("Attempt output was written but the commit pointer was not published")
        committed = True
        outcome = {
            "work_id": work_id,
            "attempt_id": attempt_id,
            "status": "SUCCEEDED",
            "telemetry_rows": telemetry_count,
            "line_count_rows": line_count,
            "processing_seconds": result.processing_seconds,
        }
    except Exception as error:
        current = load_work()
        if current["status"] == "SUCCEEDED" and current["committed_attempt_id"] == attempt_id:
            committed = True
        if not committed:
            retryable, category = classify_error(error)
            attempts_exhausted = current["attempt_count"] >= current["max_attempts"]
            next_status = "LEASE_LOST" if isinstance(error, LeaseLostError) else "RETRY_WAIT"
            if attempts_exhausted:
                next_status = "DEAD_LETTERED"
            elif not retryable:
                next_status = "TERMINAL_FAILED"
            snapshot_error = None
            if config is not None and config.result.initialized:
                try:
                    partial_telemetry, partial_line_counts = result_frames(work, config.result)
                    replace_attempt_rows(telemetry_table, partial_telemetry)
                    replace_attempt_rows(line_counts_table, partial_line_counts)
                except Exception as partial_error:
                    snapshot_error = f"; partial snapshot failed: {type(partial_error).__name__}: {partial_error}"
            recorded_error = (str(error) + (snapshot_error or ""))[:4000]
            merge_attempt(
                {
                    "status": next_status,
                    "completed_at": utc_now(),
                    "processed_frames": config.result.processed_frames if config is not None else None,
                    "processing_seconds": config.result.processing_seconds if config is not None else None,
                    "retryable": retryable,
                    "error_category": category,
                    "error_type": type(error).__name__,
                    "error_message": recorded_error,
                }
            )
            if not isinstance(error, LeaseLostError) and current["lease_owner_attempt_id"] == attempt_id:
                retry_delay_minutes = min(60, 2 ** max(0, current["attempt_count"] - 1))
                failure_source = spark_session.createDataFrame(
                    [(
                        work_id,
                        attempt_id,
                        work_capture_date,
                        next_status,
                        utc_now() + timedelta(minutes=retry_delay_minutes),
                        category,
                        type(error).__name__,
                        recorded_error,
                    )],
                    (
                        "work_id string, attempt_id string, capture_date date, status string, "
                        "not_before timestamp, error_category string, "
                        "error_type string, error_message string"
                    ),
                )
                retry_delta(
                    lambda: (
                        DeltaTable.forName(spark_session, work_table)
                        .alias("t")
                        .merge(
                            failure_source.alias("s"),
                            (
                                "t.work_id = s.work_id AND t.capture_date = s.capture_date AND "
                                "t.lease_owner_attempt_id = s.attempt_id"
                            ),
                        )
                        .whenMatchedUpdate(
                            set={
                                "status": "s.status",
                                "not_before_at": "CASE WHEN s.status = 'RETRY_WAIT' THEN s.not_before ELSE NULL END",
                                "queue_entered_at": "CASE WHEN s.status = 'RETRY_WAIT' THEN current_timestamp() ELSE t.queue_entered_at END",
                                "lease_owner_attempt_id": "NULL",
                                "lease_dispatcher_id": "NULL",
                                "lease_acquired_at": "NULL",
                                "lease_expires_at": "NULL",
                                "last_error_category": "s.error_category",
                                "last_error_type": "s.error_type",
                                "last_error_message": "s.error_message",
                            }
                        )
                        .execute()
                    )
                )
            raise
        outcome = {
            "work_id": work_id,
            "attempt_id": attempt_id,
            "status": "SUCCEEDED_AFTER_AMBIGUOUS_COMMIT",
        }
    finally:
        if staged is not None:
            staged.unlink(missing_ok=True)
        if temporary is not None:
            temporary.rmdir()

print(json.dumps(outcome, sort_keys=True))

In [ ]:
notebookutils.notebook.exit(json.dumps(outcome, sort_keys=True))